### Testing RAG Applications 📑

#### RAG Application
This application reads data about Model Context Protocol (MCP) server from internet, stores in vector stores, chunks the data with embedding and useful to answer the question about MCP while inferenced.

<img src="./img/RAG.png" width="500" height="400" style="display: block; margin: auto;">

In [1]:
# Base packages for this section
!pip install -qU langchain-core langchain-text-splitters langchain-community langchain-chroma langchain-ollama beautifulsoup4 lxml

# If you hit: deepeval requires click<8.4.0 but click 8.4.1 is installed
# run the three commands below to fix and verify dependency conflicts.
!python -m pip install "click>=8.0.0,<8.4.0"
!python -m pip install --upgrade --force-reinstall "deepeval==4.0.4"
!python -m pip check

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepeval 4.0.4 requires click<8.4.0,>=8.0.0, but you have click 8.4.1 which is incompatible.
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.6 kB)
Using cached click-8.3.3-py3-none-any.whl (110 kB)
  Attempting uninstall: click
    Found existing installation: click 8.4.1
    Uninstalling click-8.4.1:
      Successfully uninstalled click-8.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.16.4 requires click>=8.4.0, but you have click 8.3.3 which is incompatible.
  Using cached deepeval-4.0.4-py3-none-any.whl.metadata (27 kB)
  Using cached aiohttp-3.13.5-cp312-cp312-macosx_11_0_arm64.whl.metadata (8.1 kB)
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.

In [2]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_ollama import ChatOllama

/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_62097/2351560790.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen2.5:latest",
    temperature=0.5,
    max_tokens = 250
)

In [4]:
# Load data from Web
loader = WebBaseLoader("https://www.descope.com/learn/post/mcp")
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(data)

# Add text to vector db
embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever()

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}
    
    Give a summary not the full detail

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()


#### Output of the LLM Application

In [5]:
response = chain.invoke("What is MCP")

print(response)

MCP, or Model Context Protocol, standardizes how language models (LLMs) interact with external systems. It achieves this by streaming tool definitions, including their capabilities, data stores, and possible prompts, to LLMs from an MCP server. This protocol helps in integrating LLMs more effectively with various applications without requiring model-specific function schemas or complex handler implementations.


### Testing RAG Application with DeepEval
<img src="./img/RAGTesting.png" width="800" height="400" style="display: block; margin: auto;">

In [6]:
from deepeval.test_case import LLMTestCase
from deepeval.dataset import EvaluationDataset

test_case = LLMTestCase(
    input="What is MCP?",
    actual_output=chain.invoke("What is MCP"),
    expected_output="The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps"
)

dataset = EvaluationDataset()
dataset.add_test_case(test_case=test_case)

In [7]:
dataset

EvaluationDataset(test_cases=[LLMTestCase(input='What is MCP?', actual_output='MCP, or Model Context Protocol, standardizes how language models (LLMs) integrate with external systems. It achieves this by streaming tool definitions—complete with their capabilities, data stores, and possible prompts—to LLMs from an MCP server. This protocol helps in defining and handling function schemas model-agnostically, reducing the need for model-specific schema definitions and handler implementations when directly using function calls with LLM APIs.', expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps', context=None, retrieval_context=None, metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_ca

In [8]:
from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import GEval

concise_metrics = GEval(
    name = "Concise",
    criteria="Assess if the actual output remains concise while preserving all essential information.",
    
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_62097/2523086990.py:1: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


In [9]:
from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import GEval

completness_metrics = GEval(
    name = "Completeness",
    criteria="Assess whether the actual output retains all the key information from the input",
    
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_62097/1944661438.py:1: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


### Evaluation with GEval 

In [10]:
from deepeval.evaluate import evaluate
from deepeval.metrics import AnswerRelevancyMetric

evaluate(dataset.test_cases, metrics=[
    completness_metrics, 
    AnswerRelevancyMetric(),
    concise_metrics
])

✨ You're running DeepEval's latest Completeness [GEval] Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Concise [GEval] Metric! (using gpt-5.4, strict=False, async_mode=True)...

/Users/karthikkk/Downloads/aiqaDemoSource-2/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                     ┃ Average Score                ┃ Pass Rate           ┃ Total        │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━ │
│  Completeness [GEval]                       │ 0.54                         │ 100.00%             │ 1            │
│  Answer Relevancy                           │ 1.00                         │ 100.00%             │ 1            │
│  Concise [GEval]                            │ 0.82                         │ 100.00%             │ 1            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=11264222;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 6.23s | token cost: 0.013757500000000002 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Completeness [GEval]', threshold=0.5, success=True, score=0.5398115898995307, reason='The output captures the main idea that MCP standardizes integration between LLMs and external systems and reduces model-specific function schema and handler work. However, it adds specifics like streaming tool definitions, data stores, and possible prompts from an MCP server without any source input to verify against, and there is no underlying input content provided in the test case to confirm whether all key details were preserved or altered. Given the missing source input, alignment can only be judged as partial rather than fully verified.', strict_mode=False, evaluation_model='gpt-5.4', error=None, evaluation_cost=0.004110000000000001, verbose_logs='Criteria:\nAssess whether the actual output retains all the key information from the input \n \nEvaluation Steps:\n[\n    "Identify the key infor